In [ ]:
from utils import *
from data import SEED, all_classes, class_to_id, id_to_class, class_to_conf_mat, num_classes, get_loaders, get_transformer
from model import TilesModel

import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sn
from sklearn.metrics import classification_report, precision_recall_fscore_support

In [ ]:
seed_everything(SEED)

In [ ]:
test_path = '/home/linuxu/Desktop/testset3'
main_path = '/home/linuxu/Desktop/miriam'
all_paths = [ os.path.join(main_path, folder, 'separated_tiles') for folder in os.listdir(main_path) ]
nrows, ncols = 2, 4
loaders = get_loaders(test_path, all_paths, batch_size=nrows*ncols, )
imgs, lbls = next(iter(loaders['train']))

In [ ]:
fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, 10))
for i, (img, lbl) in enumerate(zip(imgs, lbls)):
    col_idx = i % ncols
    row_idx = i // ncols

    img = torch.moveaxis(img, 0, 2).numpy()
    img = np.interp(img, (img.min(), img.max()), (0, 1)).astype(np.float32)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    
    axs[ row_idx, col_idx ].imshow(img)
    axs[ row_idx, col_idx ].axis('off')

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert(torch.cuda.is_available())


In [ ]:
run_path = '/home/linuxu/Desktop/project/5_classes/runs/06-18-22_1325'

In [ ]:
model_path = os.path.join(run_path, 'models/model_f1_checkpoint.pth')
assert os.path.exists(model_path), print(model_path)

model = TilesModel(name='convnext_tiny', path=model_path)


In [ ]:
csv_paths = {
    'train': os.path.join(run_path, 'logs', 'train_data.csv'),
    'valid': os.path.join(run_path, 'logs', 'valid_data.csv'),
    'test': os.path.join(run_path, 'logs', 'test_data.csv'),
}

df_phases = {
    phase: pd.read_csv(csv_paths[phase]) for phase in ['train', 'valid', 'test']
}

In [ ]:
plot_df = pd.DataFrame(columns=[ 'phase', 'mosaic', 'tile', 'ground_truth', 'prediction', 'path' ])
infer_df = pd.DataFrame(columns=[ 'mosaic', 'tile', 'path', 'phase', 'ground_truth', 'prediction', 'prediction2','confidence',
'clod_output', 'mesh_output', 'ring_output', 'clod_mesh_output', 'none_output',
'clod_soft', 'mesh_soft', 'ring_soft', 'clod_mesh_soft', 'none_soft' ])

In [ ]:
model = model.to(device)
model.eval()
torch.autograd.set_grad_enabled(False)
transforms = get_transformer('test')

for phase in ['train', 'valid', 'test']:
    for idx, row in tqdm(    
                df_phases[phase].iterrows(), total=len(df_phases[phase]), desc=phase
            ):

        tile_path = row['x']
        truth = row['y']

        img = cv2.imread(tile_path)
        img = transforms(image=img)['image']
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = torch.from_numpy(img).float().unsqueeze(0).unsqueeze(0).to(device)

        outputs = model(img)
        outputs_soft = F.softmax(outputs.squeeze(0), dim=0)
        top_predicted_val, top_predicted_id = torch.topk(outputs_soft, k=2)

        path_split = tile_path.split('/')
        tile_name = path_split[-1]
        mosaic_name = path_split[-3]

        prediction = top_predicted_id[0].item()
        prediction2 = top_predicted_id[1].item()

        if 'prev' not in tile_path:
            plot_df = plot_df.append({ 'phase': phase, 'mosaic': mosaic_name, 'tile': tile_name, 'ground_truth': int(truth), 'prediction': top_predicted_id[0].item(), 'path': tile_path }, ignore_index=True)

        infer_df = infer_df.append({ 
            'mosaic': mosaic_name,
            'tile': tile_name,
            'path': tile_path,
            'phase': phase,

            'ground_truth': int(truth),
            'prediction': prediction,
            'prediction2': prediction2,
            'confidence': top_predicted_val[0].item() - top_predicted_val[1].item(),

            'clod_output': outputs[0][ class_to_id['clod'] ].cpu().numpy(),
            'mesh_output': outputs[0][ class_to_id['mesh'] ].cpu().numpy(),
            'ring_output': outputs[0][ class_to_id['ring'] ].cpu().numpy(),
            'clod_mesh_output': outputs[0][ class_to_id['clod_mesh'] ].cpu().numpy(),
            'none_output': outputs[0][ class_to_id['none'] ].cpu().numpy(),

            'clod_soft': outputs_soft[ class_to_id['clod'] ].cpu().numpy(),
            'mesh_soft': outputs_soft[ class_to_id['mesh'] ].cpu().numpy(),
            'ring_soft': outputs_soft[ class_to_id['ring'] ].cpu().numpy(),
            'clod_mesh_soft': outputs_soft[ class_to_id['clod_mesh'] ].cpu().numpy(),
            'none_soft': outputs_soft[ class_to_id['none'] ].cpu().numpy(),
            }, ignore_index=True)


        del img, truth, tile_path, outputs, outputs_soft, top_predicted_val, top_predicted_id
        torch.cuda.empty_cache()

In [ ]:
infer_df

In [ ]:
assert not infer_df[['mosaic', 'tile']].duplicated().any(), display(infer_df[infer_df.duplicated(['mosaic', 'tile'], keep=False)].sort_values(by='tile'))

In [ ]:
reversed_dict = { val: key for key, val in class_to_conf_mat.items() }

In [ ]:
def gen_dist_graphs_by_phase(column, to_show=False):
    if column == 'pred':
        sub_title = 'Prediction'
        column = 'prediction'
    else:
        sub_title = 'Ground Truth'
        column = 'ground_truth'

    tmp_df = plot_df.copy()
    tmp_df[column] = tmp_df[column].map(reversed_dict)

    plt.figure(figsize=(20,10), dpi=100)
    fontsize = 16
    plt.rcParams.update({'font.size': fontsize})
    ax = sn.countplot(x=column, hue='phase', data=tmp_df)
    heights = [ p.get_height() if p.get_height() >= 0 else 0 for p in ax.patches ]

    y_limit = max(heights)
    y_limit = int(np.ceil(y_limit / 100.0)) * 100
    plt.ylim([0, y_limit])

    train_sum = sum(heights[ :5 ])
    valid_sum = sum(heights[ 5:10 ])
    test_sum = sum(heights[ 10: ])

    plt.title('{} Classes Distribution'.format(sub_title))
    for i, p in enumerate(ax.patches):
        if i<5:
            sum_to_use = train_sum
        elif 5<=i<10:
            sum_to_use = valid_sum
        else:
            sum_to_use = test_sum
        if np.isnan(p.get_height()):
            ax.annotate('0(0%)', (p.get_x()+0.1, 1))
            continue
        ax.annotate('{}({:d}%)'.format(int(p.get_height()), int(p.get_height()*100/sum_to_use)), (p.get_x(), p.get_height()+1), fontsize=fontsize)

    plt.savefig(os.path.join(run_path, 'media', '{}.png'.format(column)), bbox_inches='tight')
    if to_show:
        plt.show()
    plt.close()
    del tmp_df

gen_dist_graphs_by_phase('gt')
gen_dist_graphs_by_phase('pred')

In [ ]:
def gen_confmat(to_show=False, gray=False, norm=None):
    for phase in ['train', 'valid', 'test']:

        y_true = infer_df[infer_df['phase']==phase]['ground_truth'].tolist()
        y_pred = infer_df[infer_df['phase']==phase]['prediction'].tolist()

        fig =  calc_conf_mat_img(y_true, y_pred, phase=phase, norm=norm)
        fig.savefig(os.path.join(run_path, 'media', '{}{}_conf_mat.png'.format(phase, '_norm' if norm else '')), bbox_inches='tight')
        if to_show:
            plt.show()
        plt.close()

gen_confmat(to_show=False, gray=True)
gen_confmat(to_show=False, gray=True, norm='true')

In [ ]:
plot_df

In [ ]:
assert not plot_df['tile'].duplicated().any(), display(plot_df[plot_df.duplicated(['tile'], keep=False)].sort_values(by='tile'))

In [ ]:
M = np.array([
    [0,     1,      2,     2,      2],        # none
    [1,     0,      1,     2,      2],        # ring
    [2,     1,      0,     2,      2],        # mesh (mesh / mesh_ring)
    [2,     2,      2,     0,      1],        # compo (clod_mesh / clod_mesh_ring)
    [2,     2,      2,     1,      0],        # clod / clod_ring
#   none   ring   mesh/   compo    clod/
#               mesh_ring         clod_ring
], dtype=np.float64)
M = (M / M.max())
print(M)

In [ ]:
def calc_metrics(df, phase, confidence=None, matrix=None):
    phase_df = df[df['phase']==phase]

    if not confidence:
        confidence = 0

    predicted = torch.LongTensor(phase_df[phase_df['confidence']>=confidence][['prediction']].to_numpy().astype(np.int8))
    truth = torch.LongTensor(phase_df[phase_df['confidence']>=confidence][['ground_truth']].to_numpy().astype(np.int8))

    preds = predicted.tolist()
    truth = truth.squeeze(1).tolist()

    cm = np.array(confusion_matrix(truth, preds, labels=[ class_to_conf_mat[cls] for cls in class_to_conf_mat ]))


    accuracy, recall, precision, f1 = [], [], [], []

    for cls in range(len(class_to_conf_mat)):
        tp, tn, fp, fn = 0, 0, 0, 0

        tp = cm[ cls, cls ]

        for i in range(len(class_to_conf_mat)):
            if i == cls:
                continue
            for j in range(len(class_to_conf_mat)):
                if j == cls:
                    continue
                tn += cm[ i, j ]

        for i in range(len(class_to_conf_mat)):
            if i == cls:
                continue
            fn += cm[ cls, i ] * matrix[ cls, i ] if matrix is not None else cm[ cls, i ]

        for i in range(len(class_to_conf_mat)):
            if i == cls:
                continue
            fp += cm[ i, cls ] * matrix[ cls, i ] if matrix is not None else cm[ i, cls ]
            
        acc = (tp + tn) / (tp + tn + fp + fn)
        prec = tp / (tp + fp)
        rec = tp / (tp + fn)

        accuracy.append( acc )
        precision.append( prec )
        recall.append( rec )
        f1.append( (2 * prec * rec) / (prec + rec) )
    
    met_df = pd.DataFrame( zip( id_to_class.values(), accuracy, precision, recall, f1 ), columns=[ 'class', 'accuracy', 'precision', 'recall', 'f1' ] )
    
    print('Confidence>={}'.format(confidence))
    display(met_df)

    accuracy_mean, precision_mean, recall_mean, f1_mean = 0, 0, 0, 0
    for cls_id in id_to_class:
        accuracy_mean += met_df[met_df['class']==id_to_class[cls_id]]['accuracy'].to_numpy() * np.count_nonzero(np.array(truth) == cls_id)
        precision_mean += met_df[met_df['class']==id_to_class[cls_id]]['precision'].to_numpy() * np.count_nonzero(np.array(truth) == cls_id)
        recall_mean += met_df[met_df['class']==id_to_class[cls_id]]['recall'].to_numpy() * np.count_nonzero(np.array(truth) == cls_id)
        f1_mean += met_df[met_df['class']==id_to_class[cls_id]]['f1'].to_numpy() * np.count_nonzero(np.array(truth) == cls_id)

    accuracy_mean = accuracy_mean[0] / len(truth)
    precision_mean = precision_mean[0] / len(truth)
    recall_mean = recall_mean[0] / len(truth)
    f1_mean = f1_mean[0] / len(truth)

    print('{:10s}{:.5f}'.format('Accuracy', accuracy_mean))
    print('{:10s}{:.5f}'.format('Precision', precision_mean))
    print('{:10s}{:.5f}'.format('Recall', recall_mean))
    print('{:10s}{:.5f}'.format('F1', f1_mean))

calc_metrics(infer_df, 'test', confidence=0.9, matrix=M)
calc_metrics(infer_df, 'test', confidence=0, matrix=M)


In [ ]:
def calc_top2_diff(df, phase):
    phase_df = df[df['phase']==phase]

    diff = phase_df[phase_df['prediction2']==phase_df['ground_truth']][['confidence']]
    
    fig, ax = plt.subplots(figsize=(10, 10))
    plt.rcParams.update({'font.size': 20})
    sn.histplot(ax=ax, data=diff, x='confidence', kde=True, bins=len(diff)//4)
    plt.title('Confidence Distribution')
        
    fig.tight_layout()
    plt.show()
    plt.close()

calc_top2_diff(infer_df, 'test')

In [ ]:
main_path = '/home/linuxu/Desktop/miriam'
all_paths = [ os.path.join(main_path, folder, 'separated_tiles') for folder in os.listdir(main_path) ]
test_path = '/home/linuxu/Desktop/testset3'
all_paths.append(test_path)
transforms = get_transformer('test')

not_tagged_tiles = []
for p in all_paths:
    mosaics_only = [ m for m in os.listdir(p) 
                                    if 'COMP' in m 
                                    or 'AI' in m
                                    or 'CON' in m
                                    or 'PRG' in m
                                        ] # filter out folders that are not mosaics
                                        
    for mosaic_name in mosaics_only:
        if 'testset' not in p and mosaic_name in os.listdir(test_path):
            continue
        mos_path = os.path.join(p, mosaic_name)
        
        for item in os.listdir(mos_path):
            item_path = os.path.join(mos_path, item)
            if os.path.isdir(item_path):
                continue
            not_tagged_tiles.append(item_path)

for tile_path in tqdm(not_tagged_tiles):    
    img = cv2.imread(tile_path)
    img = transforms(image=img)['image']
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = torch.from_numpy(img).float().unsqueeze(0).unsqueeze(0).to(device)

    outputs = model(img)
    predicted = F.softmax(outputs, dim=1).argmax(dim=1)


    splited_name = tile_path.split('/')
    tile_name = splited_name[-1]
    mosaic_name = splited_name[-2]
    phase = 'test' if 'testset' in tile_path else 'train'

    plot_df = plot_df.append({ 'phase': phase, 'mosaic': mosaic_name, 'tile': tile_name, 'ground_truth': -1, 'prediction': predicted.cpu().tolist()[0], 'path': tile_path  }, ignore_index=True)


    del img, outputs, predicted

In [ ]:
assert not plot_df['tile'].duplicated().any(), display(plot_df[plot_df.duplicated(['tile'], keep=False)].sort_values(by='tile'))

In [ ]:
plot_df

In [ ]:
print(len(plot_df))

In [ ]:
assert not plot_df['tile'].duplicated().any(), display(plot_df[plot_df.duplicated(['tile'], keep=False)].sort_values(by='tile'))

In [ ]:
main_path = '/home/linuxu/Desktop/miriam'
all_paths = [ os.path.join(main_path, folder, 'separated_tiles') for folder in os.listdir(main_path) ]
test_path = '/home/linuxu/Desktop/testset2'
all_paths.append(test_path)
transforms = get_transformer('test')

missing_tiles = []
for p in all_paths:
    mosaics_only = [ m for m in os.listdir(p) 
                                    if 'COMP' in m 
                                    or 'AI' in m
                                    or 'CON' in m
                                    or 'PRG' in m
                                        ] # filter out folders that are not mosaics
                                        
    for mosaic_name in mosaics_only:
        mos_path = os.path.join(p, mosaic_name)

        if 'testset' not in p and mosaic_name in os.listdir(test_path):
            continue
        
        for folder_name in os.listdir(mos_path):
            folder_path = os.path.join(mos_path, folder_name)

            if os.path.isfile(folder_path):
                continue

            for tile_name in os.listdir(folder_path):                
                if plot_df['tile'].str.contains(tile_name).any():
                    continue

                tile_path = os.path.join(folder_path, tile_name)

                missing_tiles.append(tile_path)

for tile_path in tqdm(missing_tiles):    
    img = cv2.imread(tile_path)
    img = transforms(image=img)['image']
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = torch.from_numpy(img).float().unsqueeze(0).unsqueeze(0).to(device)

    outputs = model(img)
    predicted = F.softmax(outputs, dim=1).argmax(dim=1)

    splited_name = tile_path.split('/')
    tile_name = splited_name[-1]
    ground_truth = class_to_id[ splited_name[-2] ]
    mosaic_name = splited_name[-3]
    phase = 'test' if 'testset' in tile_path else 'train'

    plot_df = plot_df.append({ 'phase': phase, 'mosaic': mosaic_name, 'tile': tile_name, 'ground_truth': ground_truth, 'prediction': predicted.cpu().tolist()[0], 'path': tile_path  }, ignore_index=True)

    del img, outputs, predicted

In [ ]:
assert(not plot_df['tile'].duplicated().any())

In [ ]:
len(plot_df)

In [ ]:
def get_numbers_from_tile(tile_name):
    tile_name = tile_name.split('.')[0]
    splited_name = tile_name.split('_')

    h_section, w_section = splited_name[1], splited_name[2]

    y, h = h_section.split('-')
    x, w = w_section.split('-')

    return int(y)-1, int(x)-1, int(h), int(w)

def get_tile_index(tile_name):
    y, x, h, w = get_numbers_from_tile(tile_name)
    return y, x

def get_mosaic_dimentions(tile_name):
    y, x, h, w = get_numbers_from_tile(tile_name)
    return h, w

In [ ]:
def get_num_neighbors_preds(df, tile_name, tile_x, tile_y, max_x, max_y, neighbors_offset, tile_prediction):
    if not max_x > tile_x > 1 or not max_y > tile_y > 1:
        return 1

    preds = []
    for x in range(tile_x-neighbors_offset, tile_x+neighbors_offset+1):
        for y in range(tile_y-neighbors_offset, tile_y+neighbors_offset+1):

            neighbor_name = '{}_{}-{}_{}-{}.bmp'.format(tile_name.split('_')[0], x, max_x, y, max_y)
            
            label = df[df['tile']==neighbor_name]['prediction'].values
            if label == class_to_id['none'] or label == 'none':
                continue
            
            preds.append( label )

    return len(np.unique(preds))

def calc_regularity(df, neighbors_offset=1):
    reg_dict = dict()
    for mosaic_name in tqdm(df['mosaic'].unique()):
        df_mos = df[df['mosaic']==mosaic_name]
        for tile_name in df_mos['tile']:
            x, y, max_x, max_y = get_numbers_from_tile(tile_name)       # the func returns x and y as an index, therfore it necessary to add 1 to each (the tile names start from 1 and not 0)
            x, y = x+1, y+1

            num = get_num_neighbors_preds(df_mos, tile_name, x, y, max_x, max_y, neighbors_offset, df_mos[df_mos['tile']==tile_name]['prediction'].values[0])
            reg_dict[tile_name] = num

    return reg_dict

In [ ]:
regularity = calc_regularity(plot_df)
plot_df['regularity'] = plot_df['tile'].map(regularity)
display(plot_df)

In [ ]:
TILE_HEIGHT, TILE_WIDTH = 100, 100

In [ ]:
def anot_tile(tile_path, pred_class, not_tagged, is_correct, regularity):

    alpha = 0.6

    colors = {
        class_to_id['clod']: [255, 0, 0],
        class_to_id['mesh']: [0, 255, 0],
        class_to_id['ring']: [0, 0, 255],
        class_to_id['clod_mesh']: [255, 255, 255],
        class_to_id['none']: [0, 0, 0]
    }

    middle_x, middle_y = TILE_WIDTH//2, TILE_HEIGHT//2

    tile = cv2.imread(tile_path)                      #   read image
    tile = cv2.resize(tile, (TILE_HEIGHT, TILE_WIDTH))
    tile = cv2.cvtColor(tile, cv2.COLOR_BGR2RGB)

    color_map = np.ones(tile.shape, np.uint8)

    if not_tagged:
        x_tile = tile.copy()
        x_tile = cv2.line(x_tile, pt1=(0, 0), pt2=(TILE_HEIGHT, TILE_WIDTH), color=(0, 0, 0), thickness=2)
        x_tile = cv2.line(x_tile, pt1=(TILE_HEIGHT, 0), pt2=(0, TILE_WIDTH), color=(0, 0, 0), thickness=2)
        x_alpha = 0.6
        tile = cv2.addWeighted(x_tile, 1 - x_alpha, tile, x_alpha, 0, x_tile)

    if regularity:
        o_radius = 15
        o_color = (255, 0, 0)
        o_tile = tile.copy()
        o_tile = cv2.circle(o_tile, (middle_x, middle_y), radius=o_radius, color=o_color, thickness=2)
        o_alpha = 0.
        tile = cv2.addWeighted(o_tile, 1 - o_alpha, tile, o_alpha, 0, o_tile)

    if pred_class == class_to_id['none']: # none
        color_tile = cv2.cvtColor(tile, cv2.COLOR_RGB2BGR)
        return color_tile

    color_map = cv2.rectangle(img=color_map, pt1=(0, 0), pt2=(color_map.shape[0], color_map.shape[1]), color=colors[ pred_class ], thickness=cv2.FILLED)

    color_tile = tile.copy()
    mask = color_map.astype(bool)
    color_tile[mask] = cv2.addWeighted(tile, alpha, color_map, 1 - alpha, 0)[mask]
    color_tile = cv2.cvtColor(color_tile, cv2.COLOR_RGB2BGR)
    return color_tile

def anot_mosaic(df, max_regularity=3):

    h_mosaic, w_mosaic = get_mosaic_dimentions( df.iloc[0]['tile'] )
    mos = np.ones((h_mosaic*TILE_HEIGHT, w_mosaic*TILE_WIDTH, 3))
    
    for idx, row in df.iterrows():
        tile_path = row['path']
        pred_class = row['prediction']
        not_tagged = row['ground_truth'] == 'not_tagged' or row['ground_truth'] == -1
        is_correct = row['ground_truth'] == row['prediction']
        colored_tile = anot_tile(tile_path, pred_class, not_tagged, is_correct, row['regularity'] >= max_regularity)

        y, x = get_tile_index( row['tile'] )
        
        h_start, w_start = y * TILE_HEIGHT, x * TILE_WIDTH
        h_end, w_end = h_start + TILE_HEIGHT, w_start + TILE_WIDTH

        mos[ h_start : h_end, w_start : w_end, : ] = colored_tile

    return mos

In [ ]:
def anot_tile_regularity(tile_path, regularity):

    middle_x, middle_y = TILE_WIDTH//2, TILE_HEIGHT//2

    tile = cv2.imread(tile_path)                      #   read image
    tile = cv2.resize(tile, (TILE_HEIGHT, TILE_WIDTH))
    tile = cv2.cvtColor(tile, cv2.COLOR_BGR2RGB)

    if regularity:
        o_radius = 15
        o_color = (255, 0, 0)
        o_tile = tile.copy()
        o_tile = cv2.circle(o_tile, (middle_x, middle_y), radius=o_radius, color=o_color, thickness=2)
        o_alpha = 0.1
        tile = cv2.addWeighted(o_tile, 1 - o_alpha, tile, o_alpha, 0, o_tile)
        tile = cv2.cvtColor(tile, cv2.COLOR_RGB2BGR)

    return tile

def anot_mosaic_regularity(df, max_regularity=3):

    h_mosaic, w_mosaic = get_mosaic_dimentions( df.iloc[0]['tile'] )
    mos = np.ones((h_mosaic*TILE_HEIGHT, w_mosaic*TILE_WIDTH, 3))
    
    for idx, row in df.iterrows():
        tile_path = row['path']
        pred_class = row['prediction']
        not_tagged = row['ground_truth'] == 'not_tagged' or row['ground_truth'] == -1
        is_correct = row['ground_truth'] == row['prediction']
        colored_tile = anot_tile_regularity(tile_path, row['regularity'] >= max_regularity)

        y, x = get_tile_index( row['tile'] )
        
        h_start, w_start = y * TILE_HEIGHT, x * TILE_WIDTH
        h_end, w_end = h_start + TILE_HEIGHT, w_start + TILE_WIDTH

        mos[ h_start : h_end, w_start : w_end, : ] = colored_tile

    return mos

In [ ]:
main_save_path = os.path.join(run_path, 'media', 'mosaics')
if not os.path.exists(main_save_path):
    os.mkdir(main_save_path)

for phase in ['test', 'train']:
    phase_save_path = os.path.join(main_save_path, phase)
    if not os.path.exists(phase_save_path):
        os.mkdir(phase_save_path)

    for mos in tqdm( plot_df[plot_df['phase']==phase]['mosaic'].unique() ):
        anot_mos = anot_mosaic( plot_df[plot_df['mosaic']==mos] )
        save_path = os.path.join(phase_save_path, mos + '_predicted.bmp')
        cv2.imwrite(save_path, anot_mos)

        anot_mos_reg = anot_mosaic_regularity( plot_df[plot_df['mosaic']==mos] )
        save_path_reg = os.path.join(phase_save_path, mos + '_regularity.bmp')
        cv2.imwrite(save_path_reg, anot_mos_reg)
        del anot_mos, anot_mos_reg

In [ ]:
infer_df.to_csv(os.path.join(run_path, 'logs', 'infer.csv'))
plot_df.to_csv(os.path.join(run_path, 'logs', 'plot.csv'))

In [ ]:
infer_df = pd.read_csv(os.path.join(run_path, 'logs', 'infer.csv'))
plot_df = pd.read_csv(os.path.join(run_path, 'logs', 'plot.csv'))

In [ ]:
def get_mosaic_percentage(mos_name):
    mos_df = plot_df[plot_df['mosaic']==mos_name]

    return len(mos_df[mos_df['ground_truth']!=-1]) * 100 / len(mos_df)

def get_mosaic_accuracy(mos_name):
    mos_df = infer_df[infer_df['mosaic']==mos_name]

    if mos_df.empty:
        return None

    outputs = torch.FloatTensor(mos_df[['none_output', 'ring_output', 'mesh_output', 'clod_mesh_output', 'clod_output']].to_numpy(dtype=np.float32))
    truth = torch.LongTensor(mos_df[['ground_truth']].to_numpy())

    predicted = F.softmax(outputs, dim=1).argmax(dim=1)

    preds = torch.LongTensor(predicted)
    truth = torch.LongTensor(truth).squeeze(1)
    
    return torchmetrics.functional.accuracy(preds, truth, average='micro').item() * 100

In [ ]:
def annotate_predictions(preds_path):
    anbot_mosaic_path = '/home/linuxu/Desktop/project/5_classes/anot_mosaics'

    mosaics_to_show = [ m for m in os.listdir(preds_path) if 'predicted' in m ]
    for idx, mosaic in enumerate(tqdm(mosaics_to_show)):
        if 'predicted' not in mosaic:
            continue

        mosaic_name = mosaic.split('_')[0]
        mos_name_wo_ext = mosaic_name.split('.')[0]
        save_path = os.path.join(preds_path, '{}_gt_vs_pred.png'.format(mos_name_wo_ext))

        pred_path = os.path.join(preds_path, mosaic)
        
        mosaic_name = mosaic.split('_')[0]
        anot_name = mosaic_name + '_anot.' + 'bmp'
        anot_path = os.path.join(anbot_mosaic_path, anot_name)

        pred_mos = cv2.imread(pred_path)
        pred_mos = cv2.cvtColor(pred_mos, cv2.COLOR_BGR2RGB)

        anot_mos = cv2.imread(anot_path)
        anot_mos = cv2.cvtColor(anot_mos, cv2.COLOR_BGR2RGB)
        
        h, w, _ = pred_mos.shape
        factor = 100

        fig, axs = plt.subplot_mosaic([['left', 'right']], sharey=True, figsize=(w//factor * 2, h//factor + 1))

        axs['right'].set_title('Prediction', fontsize=20)
        axs['right'].imshow(norm(pred_mos))
        axs['right'].axis('off')
        axs['left'].set_title('Ground Truth', fontsize=20)
        axs['left'].imshow(norm(anot_mos))
        axs['left'].axis('off')

        colors = {
            class_to_id['clod']: [1, 0, 0],
            class_to_id['mesh']: [0, 1, 0],
            class_to_id['ring']: [0, 0, 1],
            class_to_id['clod_mesh']: [1, 1, 1],
            class_to_id['none']: [0.5, 0.5, 0.5]
        }

        plt_handlers = []
        for i, label_name in enumerate(class_to_conf_mat.keys()):
            fc = colors[ class_to_conf_mat[label_name] ]

            p = plt.Rectangle((0, 0), 3, 3, edgecolor='black', fc=fc, label=label_name)
            plt_handlers.append(p)

        # add black 'x' to the legend
        p = plt.Line2D([0],[0], linewidth=0, color='black', marker='x', markerfacecolor='black', markersize=20, markeredgewidth=2, label='not_tagged')
        plt_handlers.append(p)

        # add red 'o' to the legend
        p = plt.Line2D([0],[0], linewidth=0, color='w', marker='o', markerfacecolor='r', markersize=20, markeredgewidth=2, label='irregularity')
        plt_handlers.append(p)
        
        ncols = len(plt_handlers)
        if plt.gcf().get_size_inches()[0]*fig.dpi < 1800:
            ncols = 5
        if plt.gcf().get_size_inches()[0]*fig.dpi < 1400:
            ncols = 4
        if plt.gcf().get_size_inches()[0]*fig.dpi < 1200:
            ncols = 3
        
        fig.legend(handles=plt_handlers, framealpha=1, ncol=ncols, bbox_to_anchor=(1, 0), fontsize=20)

        percentage = get_mosaic_percentage(mos_name_wo_ext)
        accuracy = get_mosaic_accuracy(mos_name_wo_ext)
        if accuracy:
            fig.suptitle('{} {:.2f}% Tagged\n{:.2f} Accuracy\n'.format(mos_name_wo_ext, percentage, accuracy), fontsize=30)
        else:
            fig.suptitle('{} {:.2f}% Tagged'.format(mos_name_wo_ext, percentage), fontsize=30)

        plt.tight_layout()
        save_path = os.path.join(preds_path, '{}_gt_vs_pred.png'.format(mos_name_wo_ext))
        plt.savefig(save_path, bbox_inches='tight')
        plt.close()
        del pred_mos, anot_mos

In [ ]:
annotate_predictions(os.path.join(run_path, 'media', 'mosaics', 'train'))

In [ ]:
annotate_predictions(os.path.join(run_path, 'media', 'mosaics', 'test'))